# Strategy Correlation Matrix

Pairwise return correlation across every strategy's genuine backtest equity curve under `outputs/<strategy_id>/`, plus the S&P 500 (SPY) benchmark where available. This notebook only ever reads real, already-computed equity curves -- it never fabricates, backfills, or simulates a return series for a strategy that hasn't actually been backtested.

**Extending this as new strategies are added:** write a strategy's backtest equity curve to `outputs/<strategy_id>/<strategy_id>_equity_curve.csv` with a `quarter_end` or `month_end` date column (whichever matches that strategy's own rebalance cadence) and a `<strategy_id>_equity` column. This notebook auto-discovers every `outputs/*/*_equity_curve.csv` file on each run -- no code change needed here to add a strategy to the matrix, and different strategies may use different rebalance frequencies (each file's own returns are computed on its own native cadence *before* merging, so a monthly strategy is never compared against interpolated or gap-filled quarterly points).

In [ ]:
from pathlib import Path
import sys


def resolve_project_root() -> Path:
    candidates = []
    try:
        candidates.append(Path.cwd())
    except FileNotFoundError:
        pass
    candidates.append(Path("/Users/gavinhussey/Downloads/AtlasQuant"))
    for candidate in candidates:
        if candidate.name == "outputs":
            candidate = candidate.parent
        if (candidate / "src" / "atlas_quant").exists():
            return candidate
    raise FileNotFoundError("Could not locate AtlasQuant project root")


PROJECT_ROOT = resolve_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

OUTPUT_DIR = PROJECT_ROOT / "outputs"
MATRIX_CSV = OUTPUT_DIR / "strategy_correlation_matrix.csv"

# Every file here is treated as one strategy's genuine backtest equity curve,
# never a placeholder. Add a strategy to this matrix by writing
# outputs/<strategy_id>/<strategy_id>_equity_curve.csv with a "quarter_end"
# or "month_end" column and a "<strategy_id>_equity" column -- no code
# change needed below.
EQUITY_CURVE_CSVS = sorted(OUTPUT_DIR.glob("*/*_equity_curve.csv"))
DATE_COLUMN_CANDIDATES = ("quarter_end", "month_end")
BENCHMARK_COLUMN = "sp500_equity"
BENCHMARK_LABEL = "S&P 500 (SPY)"

print("project root:", PROJECT_ROOT)
print("equity curve files found:")
for p in EQUITY_CURVE_CSVS:
    print(" -", p.name)


In [ ]:
import pandas as pd

if not EQUITY_CURVE_CSVS:
    raise FileNotFoundError(
        f"no *_equity_curve.csv files found under {OUTPUT_DIR} -- "
        "run a strategy's backtest and export its equity curve first."
    )

return_frames = []
benchmark_returns = None

for csv_path in EQUITY_CURVE_CSVS:
    df = pd.read_csv(csv_path)
    date_column = next((c for c in DATE_COLUMN_CANDIDATES if c in df.columns), None)
    if date_column is None:
        raise ValueError(
            f"{csv_path.name} has none of {DATE_COLUMN_CANDIDATES} -- add one matching "
            "this strategy's own rebalance cadence"
        )
    df[date_column] = pd.to_datetime(df[date_column])
    df = df.set_index(date_column).sort_index()

    equity_columns = [c for c in df.columns if c.endswith("_equity") and c != BENCHMARK_COLUMN]
    if not equity_columns:
        raise ValueError(f"{csv_path.name} has no '<strategy_id>_equity' column to include")
    # Each file's own returns are computed on its own native index *before*
    # merging with any other file -- a strategy's monthly cadence must
    # never be diffed against an adjacent row that actually belongs to a
    # different strategy's quarterly cadence.
    for col in equity_columns:
        return_frames.append(df[col].pct_change().rename(col))

    if BENCHMARK_COLUMN in df.columns and benchmark_returns is None:
        # Every strategy backtest resolves SPY the same way (see
        # atlas_quant.backtest.benchmark) -- the first file's benchmark
        # column is used; later files are expected to agree, not re-merged.
        benchmark_returns = df[BENCHMARK_COLUMN].pct_change().rename(BENCHMARK_COLUMN)

frames = return_frames + ([benchmark_returns] if benchmark_returns is not None else [])
returns = pd.concat(frames, axis=1).sort_index()
returns = returns.dropna(how="all")
returns.tail()


In [ ]:
label_map = {
    c: (BENCHMARK_LABEL if c == BENCHMARK_COLUMN else c.removesuffix("_equity"))
    for c in returns.columns
}
returns_labeled = returns.rename(columns=label_map)

# pandas .corr() uses pairwise-complete observations by default -- exactly
# what's needed when two strategies' return series don't share every date
# (different rebalance cadences, different backtest start dates).
correlation = returns_labeled.corr()
correlation.to_csv(MATRIX_CSV)
print(f"wrote {MATRIX_CSV}")
for col in returns_labeled.columns:
    print(f"  {col}: {returns_labeled[col].notna().sum()} observations")
correlation


**Caveat:** a period where a strategy held no positions yet (e.g. Filing Momentum ML's 2015-2016 ramp-up, before enough training history existed) or falls beyond its backtest's current end date reports a flat 0.0 return in the source CSV. Those are genuine reported values, not fabricated, but simultaneous (0.0, 0.0) pairs across series mechanically push correlation *up* rather than reflecting real co-movement. The per-series observation counts printed above are the full counts including these; narrow a series' own equity-curve CSV to its active window first if a stricter, ramp-up-excluded correlation is needed. Because each file's returns are computed on its own native cadence, a monthly-rebalance strategy compared against a quarterly-rebalance one is correlated only over the dates both series actually report a return for (pairwise-complete), not over an interpolated common grid.

In [ ]:
from html import escape
from IPython.display import HTML, display


def _color_for(value: float) -> str:
    # Diverging scale: -1 -> blue, 0 -> white, +1 -> red.
    v = max(-1.0, min(1.0, value))
    if v >= 0:
        r = 255 + v * (220 - 255)
        g = 255 + v * (38 - 255)
        b = 255 + v * (38 - 255)
    else:
        t = -v
        r = 255 + t * (37 - 255)
        g = 255 + t * (99 - 255)
        b = 255 + t * (235 - 255)
    return f"rgb({r:.0f},{g:.0f},{b:.0f})"


def render_correlation_heatmap(corr: pd.DataFrame) -> str:
    labels = list(corr.columns)
    header = "".join(f'<th style="padding:8px 14px;text-align:right;font-size:13px;color:#374151;">{escape(l)}</th>' for l in labels)
    rows_html = []
    for row_label in labels:
        cells_html = []
        for col_label in labels:
            v = corr.loc[row_label, col_label]
            bg = _color_for(v)
            text_color = "#111827" if abs(v) < 0.6 else "#ffffff"
            cells_html.append(
                f'<td style="padding:8px 14px;text-align:center;background:{bg};color:{text_color};font-variant-numeric:tabular-nums;">{v:+.2f}</td>'
            )
        rows_html.append(
            f'<tr><th style="padding:8px 14px;text-align:left;font-size:13px;color:#374151;">{escape(row_label)}</th>{"".join(cells_html)}</tr>'
        )
    return f'''
    <table style="border-collapse:collapse;font-family:-apple-system,Helvetica,Arial,sans-serif;border:1px solid #d1d5db;">
      <thead><tr><th></th>{header}</tr></thead>
      <tbody>{"".join(rows_html)}</tbody>
    </table>
    '''


display(HTML(render_correlation_heatmap(correlation)))
